In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
# from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def parse_description(df): 
    KEYWORDS = {
        "duong_lon": [
            "đường lớn", "đại lộ", "mặt đường", "mặt tiền", "trục chính",
            "đường huyết mạch", "đường 8m", "đường 10m", "đường 12m",
            "đường 20m", "cao tốc", "quốc lộ", "trục đường chính"
        ],
        "vi_tri_dat": [
            "góc", "ngã tư", "ngã ba", "căn góc"
        ],
        "garage": [
            "gara", "garage", "đỗ xe"
        ],
        "san_vuon": [
            "sân vườn", "sân thượng", "ban công"
        ],
        "hai_mat_tien": [
            "2 mặt tiền", "hai mặt tiền", "3 mặt tiền"
        ],
        "view": [
            "view"
        ],
        "vi_tri_trung_tam": [
            "trung tâm", "vip", "cao cấp", "quận 1", "lõi đô thị", "quận 3", "bến thành"
        ],
        "truong_hoc": [
            "trường học", "đại học", "trường", "giáo dục"
        ],
        "san_bay": [
            "sân bay"
        ],
        "benh_vien": [
            "bệnh viện", "phòng khám", "y tế"
        ],
        "sieu_thi": [
            "siêu thị", "chợ", "mall", "shopping", "trung tâm thương mại",
            "vincom", "aeon mall", "lotte"
        ],
        "giao_thong": [
            "metro", "bến xe", "bus", "trạm sạc"
        ],
        "giai_tri": [
            "công viên", "thể thao", "khu vui chơi", "hồ bơi", "sân bóng"
        ],
        "dia_diem_dac_biet": [
            "phú mỹ hưng", "thủ thiêm", "sala", "vinhomes",
            "thảo điền", "an phú", "landmark", "saigon pearl"
        ]
    }
    PATTERNS = {
        name: re.compile(  
            r"(" + "|".join(re.escape(term) for term in terms) + r")",
            flags=re.IGNORECASE 
        )
        for name, terms in KEYWORDS.items()
    }
    for name, pattern in PATTERNS.items():
        df[name] = df["mo_ta"].fillna("").str.contains(pattern, regex=True)

    df.drop('mo_ta', axis = 1, inplace = True)
    
    return df 

Làm sạch cột số

In [ ]:
# Chuyển đổi các chuỗi thành số
def extract_number(text):
    if pd.isna(text): return np.nan
    res = re.findall(r'\d+\.?\d*', str(text).replace(',', '.'))
    return float(res[0]) if res else np.nan

def convert_price(price_str):
    if pd.isna(price_str): return np.nan
    price_str = str(price_str).lower().replace(',', '.')
    val = extract_number(price_str)
    if 'tỷ' in price_str: return val * 1e9
    if 'triệu' in price_str: return val * 1e6
    return val

In [ ]:
# chuyển đổi
def cleaning_numeric_col(df):
    cols_price=['gia_ban']
    for col in cols_price:
        df[col] = df[col].apply(convert_price)

    cols_numeric = ['dien_tich_dat', 'so_phong_ngu', 'so_phong_ve_sinh', 'tong_so_tang', 'chieu_ngang']
    for col in cols_numeric:
        df[col] = df[col].apply(extract_number)
    return df 

In [ ]:
# Tạo log các cột số 
def log_numeric_col(df):
    df['log_gia_vnd'] = np.log1p(df['gia_ban'])
    df['log_dien_tich_dat'] = np.log1p(df['dien_tich_dat'])
    df['log_gia_m2_tham_khao'] = np.log1p(df['gia_m2_tham_khao'])
    df['log_chieu_ngang'] = np.log1p(df['chieu_ngang'])

    df['price_m2'] = (df['gia_ban'] / df['dien_tich_dat'])/ 1e6 
    df['log_price_m2'] = np.log1p(df['price_m2'])

    return df 

In [ ]:
# df_final = df[['log_price_m2', 'log_dien_tich_dat']]

In [ ]:
'''
import joblib
kmeans = joblib.load("models/kmeans.pkl") # Chỗ này load lên lại rồi dùng df_final đã được xử lý để predict 
labels = kmeans.predict(X_new)
''' 